# Qwen-Image-2.1-Turbo (4-bit) API Server untuk Pixorabooth
Jalankan **Cell 1** lalu **Cell 2**, lalu salin `IMAGE_API_URL` ke `.env`.

In [ ]:
!nvidia-smi
!pip install -q -U git+https://github.com/huggingface/diffusers.git
!pip install -q -U "transformers>=5.17.0" accelerate bitsandbytes pillow sentencepiece fastapi uvicorn
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

In [ ]:
import base64, gc, re, subprocess, threading, time
from io import BytesIO
import torch, uvicorn
from diffusers import QwenImage21Pipeline
from fastapi import FastAPI, Request
from PIL import Image

if "tunnel_proc" in globals() and tunnel_proc is not None:
    try: tunnel_proc.terminate()
    except Exception: pass
for _v in ("pipe_enc", "pipe"):
    globals().pop(_v, None)
gc.collect(); torch.cuda.empty_cache()

vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
repo_id = "addlabsviral/Qwen-Image-2.1-Turbo-4bit"
print(f"GPU: {torch.cuda.get_device_name(0)} ({vram_gb:.1f} GB VRAM), dtype: {dtype}")

if vram_gb >= 20:
    pipe = QwenImage21Pipeline.from_pretrained(repo_id, torch_dtype=dtype, device_map="cuda")
else:
    pipe = QwenImage21Pipeline.from_pretrained(repo_id, torch_dtype=dtype)
    pipe.enable_model_cpu_offload()
if hasattr(pipe.vae, "enable_tiling"):
    pipe.vae.enable_tiling()
print("Model ready!")

gen_lock = threading.Lock()
api = FastAPI()

@api.get("/health")
def health():
    return {"status": "ok", "model": repo_id, "gpu": torch.cuda.get_device_name(0)}

@api.post("/v1/chat/completions")
async def chat_completions(request: Request):
    body = await request.json()
    extra = body.get("extra_body") or {}
    prompts, input_img = [], None
    for msg in body.get("messages", []):
        c = msg.get("content", "")
        if isinstance(c, str):
            prompts.append(c)
        elif isinstance(c, list):
            for p in c:
                if not isinstance(p, dict): continue
                if p.get("type") == "text" and p.get("text"):
                    prompts.append(p["text"])
                elif p.get("image_url"):
                    v = p["image_url"]
                    u = v.get("url") if isinstance(v, dict) else v
                    if u and u.startswith("data:image/"):
                        input_img = Image.open(BytesIO(base64.b64decode(u.split(",", 1)[1]))).convert("RGBA")
    prompt = "\n".join(prompts).strip() or "A vibrant artistic photograph"
    w = max(256, (int(extra.get("width", 768)) // 16) * 16)
    h = max(256, (int(extra.get("height", 512)) // 16) * 16)
    with gen_lock:
        kw = {"prompt": prompt, "width": w, "height": h, "use_kv_cache": True,
              "generator": torch.Generator("cpu").manual_seed(int(time.time()) % 100000)}
        if input_img is not None:
            kw["image"] = input_img
        img = pipe(**kw).images[0].convert("RGB")
        torch.cuda.empty_cache()
    buf = BytesIO()
    img.save(buf, format="JPEG", quality=90)
    b64 = base64.b64encode(buf.getvalue()).decode("ascii")
    return {"id": f"qwen21-{int(time.time())}", "object": "chat.completion",
            "choices": [{"index": 0, "message": {"role": "assistant",
            "content": [{"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}}]},
            "finish_reason": "stop"}]}

if "uvicorn_thread" not in globals() or not uvicorn_thread.is_alive():
    uvicorn_thread = threading.Thread(target=lambda: uvicorn.run(api, host="127.0.0.1", port=8000, log_level="warning"), daemon=True)
    uvicorn_thread.start()
time.sleep(2)

tunnel_proc = subprocess.Popen(["/usr/local/bin/cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
public_url = None
t0 = time.time()
while time.time() - t0 < 30:
    line = tunnel_proc.stdout.readline()
    if not line: break
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m:
        public_url = m.group(0); break

print("GPU:", torch.cuda.get_device_name(0))
print("HEALTH URL:", f"{public_url}/health")
print("IMAGE_API_URL:", f"{public_url}/v1/chat/completions")